<a href="https://colab.research.google.com/github/gianlucamajor/eCruziDB/blob/main/eda/agrs-eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Colab-**eCruzidb** Extended Data Analysis
## AGRs EDA

In [28]:
#@title Install dependencies and loading libs
%time
import os
from io import  StringIO
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec
from scipy.stats import zscore


CPU times: user 3 µs, sys: 0 ns, total: 3 µs
Wall time: 4.29 µs


In [29]:
#@title Download data from eCruzidb
# The authority will not require after eCruzidb became open
%time
PE_json_url = "https://projetos.lbi.iq.usp.br/trypanosoma/ecruzidb/data/epitopes-data.json"
br_a4_chrs_url = "https://projetos.lbi.iq.usp.br/trypanosoma/ecruzidb/data/GCA_015033625.1-Br-A4-chromosomes.tsv"
user = "ecruzidb"
password = "setulab"

def PE_data_handler(data):
  df = pd.DataFrame(data)
  # Adjusting columns names
  df = df.rename(columns={"ID":"PE_ID",
                      "Number of Genomic Regions":"Number_of_AGRs",
                      "Number of Peptides":"Number_of_Peptides",
                      "Number of Inserts":"Number_of_Inserts",
                      "Number of Inserts by Group":"Number_of_Inserts_by_Clinical_Group",
                      "Epitope":"PE_Sequence",
                      "Genomic Region Locus":"Locus_of_AGRs"
                      }).copy()
  df['GAGR-ID'] = df['PE_ID'].apply(lambda x: x.split('-')[0]) # The GAGR could has zero, one or more PE.
  return df



# Predicted Epitope Data
PE_response = requests.get(PE_json_url, auth=(user, password))
if PE_response.status_code == 200:
    data = PE_response.json()
    PEs_df = PE_data_handler(data)

else:
    print(f"Failed to fetch data: {PE_response.status_code}")

# Br-A4 chromosomes data
br_a4_chrs_response = requests.get(br_a4_chrs_url, auth=(user, password))
if br_a4_chrs_response.status_code == 200:
    tsv_data = StringIO(br_a4_chrs_response.text)
    br_a4_chrs_df = pd.read_csv(tsv_data, sep="\t")

else:
    print(f"Failed to fetch data: {br_a4_chrs_response.status_code}")



CPU times: user 2 µs, sys: 0 ns, total: 2 µs
Wall time: 4.29 µs


In [ ]:
#@title Preprocessing chromosomes data
br_a4_chrs_df

In [30]:
#@title Prepare dataframe
PEs_distinct_AGRs_df = PEs_df.drop_duplicates(subset=['GAGR-ID']).copy() # Keep just distinct AGRs Locus
PEs_distinct_AGRs_df
br_a4_chrs_df
# https://www.ncbi.nlm.nih.gov/datasets/genome/GCA_015033625.1/



,Assembly Accession,Assembly-unit name,Chromosome name,GC Count,GC Percent,GenBank seq accession,Molecule type,Ordering,RefSeq seq accession,Role,Seq length,UCSC style name,Unlocalized Count,Sequence name
0,GCA_015033625.1,Primary Assembly,1,1353675.0,49.5,CM026583.1,Chromosome,NaN,NaN,assembled-molecule,2738928,NaN,NaN,TcBrA4_Contig3
1,GCA_015033625.1,Primary Assembly,2,1041585.0,52.5,CM026584.1,Chromosome,NaN,NaN,assembled-molecule,1986034,NaN,NaN,TcBrA4_Contig6
2,GCA_015033625.1,Primary Assembly,3,866624.0,49.0,CM026585.1,Chromosome,NaN,NaN,assembled-molecule,1768708,NaN,NaN,TcBrA4_Contig41
3,GCA_015033625.1,Primary Assembly,4,823674.0,49.0,CM026586.1,Chromosome,NaN,NaN,assembled-molecule,1676910,NaN,NaN,TcBrA4_Contig121
4,GCA_015033625.1,Primary Assembly,5,746250.0,50.0,CM026587.1,Chromosome,NaN,NaN,assembled-molecule,1492459,NaN,NaN,TcBrA4_Contig53
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
397,GCA_015033625.1,Primary Assembly,Un,NaN,NaN,WNWZ01000370.1,Chromosome,NaN,NaN,unplaced-scaffold,15106,NaN,NaN,TcBrA4_Contig98
398,GCA_015033625.1,Primary Assembly,Un,NaN,NaN,WNWZ01000383.1,Chromosome,NaN,NaN,unplaced-scaffold,26682,NaN,NaN,TcBrA4_Contig175
399,GCA_015033625.1,Primary Assembly,Un,NaN,NaN,WNWZ01000385.1,Chromosome,NaN,NaN,unplaced-scaffold,26168,NaN,NaN,TcBrA4_Contig331
400,GCA_015033625.1,Primary Assembly,Un,NaN,NaN,WNWZ01000386.1,Chromosome,NaN,NaN,unplaced-scaffold,103134,NaN,NaN,TcBrA4_Contig399
